# Download Foreign/International Investor Ownership Data

Downloads foreign investor ownership data for IDX stocks from Refinitiv using `rd.get_data()`.

**Goal:** Get % of shares held by international (non-Indonesian) investors for each stock over time.

**Approach:** Test multiple Refinitiv field/parameter combos to find what works, then batch download.

**Run order:** Cell 1 (setup) -> 2 (config) -> 3 (diagnostic) -> update fields if needed -> 4 (functions) -> 5 (download) -> 6 (retry) -> 7 (combine) -> 8 (save) -> 9 (quality check)

In [1]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
from datetime import datetime
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

rd.open_session()

<refinitiv.data.session.Definition object at 0x11fd58260 {name='workspace'}>

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────
BATCH_SIZE  = 10
SLEEP       = 2
MAX_RETRIES = 3
CHUNK_PREFIX = 'inst_own_chunk'
DATA_DIR    = '../data'

# 7 chunks: 2008–2025 (matching prices and fundamentals)
CHUNKS = [
    (datetime(2008, 1, 1),  datetime(2009, 12, 31)),
    (datetime(2010, 1, 1),  datetime(2011, 12, 31)),
    (datetime(2012, 1, 1),  datetime(2014, 12, 31)),
    (datetime(2015, 1, 1),  datetime(2017, 12, 31)),
    (datetime(2018, 1, 1),  datetime(2020, 12, 31)),
    (datetime(2021, 1, 1),  datetime(2023, 12, 31)),
    (datetime(2024, 1, 1),  datetime(2025, 12, 31)),
]

# Ownership fields — StatType=7 gives Foreign/Domestic/Institutions breakdown
INST_FIELDS = [
    'TR.CategoryOwnershipPct.date',
    'TR.CategoryOwnershipPct',
    'TR.InstrStatCatSharesHeld',
    'TR.CategoryInvestorCount',
    'TR.InstrStatTypeValue',
]

INST_PARAMS_BASE = {'StatType': 7, 'Frq': 'M'}

# Load RIC list
all_rics = pd.read_csv(f'{DATA_DIR}/idx_ric_list.csv')['RIC'].tolist()

n_batches = (len(all_rics) + BATCH_SIZE - 1) // BATCH_SIZE
print(f'RICs to download: {len(all_rics)}')
print(f'Batches per chunk: {n_batches}')
print(f'Total API calls: {n_batches * len(CHUNKS)}')
print(f'Estimated time: ~{n_batches * len(CHUNKS) * (SLEEP + 1) / 60:.0f} min')

RICs to download: 992
Batches per chunk: 100
Total API calls: 500
Estimated time: ~25 min


In [3]:
def fetch_inst_batch(rics, start, end, retries=MAX_RETRIES):
    """Download ownership breakdown for a batch of RICs with retries."""
    params = {
        **INST_PARAMS_BASE,
        'SDate': start.strftime('%Y-%m-%d'),
        'EDate': end.strftime('%Y-%m-%d'),
    }
    for attempt in range(1, retries + 1):
        try:
            df = rd.get_data(
                universe=rics,
                fields=INST_FIELDS,
                parameters=params
            )
            if df is not None and not df.empty:
                return df, None
            else:
                return None, None
        except Exception as e:
            err = str(e)[:150]
            if attempt < retries:
                wait = SLEEP * (2 ** attempt)
                time.sleep(wait)
            else:
                return None, err
    return None, 'max retries exceeded'


def clean_inst_batch(df):
    """Clean raw ownership DataFrame from StatType=7.
    
    Expected 6 columns:
      Instrument, Date, Category Percent Of Traded Shares,
      Category Investor Shares Held, Category Number of Investors,
      Investor Statistics Category Value
    """
    expected = 6
    if len(df.columns) != expected:
        print(f'    WARNING: expected {expected} cols, got {len(df.columns)}: {list(df.columns)}')
        return None
    
    df = df.copy()
    df.columns = ['Instrument', 'Date', 'Pct_Held', 'Shares_Held', 'Num_Investors', 'Category']
    
    # Type conversions
    for col in ['Pct_Held', 'Shares_Held', 'Num_Investors']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    
    # Drop rows with no date
    df = df.dropna(subset=['Date'])
    
    return df


print('Functions defined.')

Functions defined.


## Diagnostic: test one batch first

Run this cell first to verify the fields return data and check the column structure before launching the full download.

In [4]:
# ── Diagnostic: test fields for foreign vs domestic ownership ───────────
test_rics = ['BBCA.JK', 'BBRI.JK', 'TLKM.JK', 'ASII.JK', 'UNVR.JK']

# Approach 1: Category stats by Location (StatType=3)
# Should give domestic vs foreign breakdown
print('=== Approach 1: CategoryOwnershipPct by Location (StatType=3) ===')
try:
    df1 = rd.get_data(
        universe=test_rics,
        fields=[
            'TR.CategoryOwnershipPct.date',
            'TR.CategoryOwnershipPct',
            'TR.InstrStatCatSharesHeld',
            'TR.CategoryInvestorCount',
            'TR.InstrStatTypeValue',
        ],
        parameters={'StatType': 3, 'SDate': '2020-01-01', 'EDate': '2020-12-31', 'Frq': 'M'}
    )
    print(f'Shape: {df1.shape}')
    print(f'Columns: {list(df1.columns)}')
    print(f'\nUnique categories (InstrStatTypeValue):')
    print(df1.iloc[:, -1].value_counts().to_string())
    print(f'\nNon-null counts:')
    print(df1.notna().sum().to_string())
    print(f'\nSample rows:')
    display(df1.head(30))
except Exception as e:
    print(f'FAILED: {e}')

print('\n' + '='*65 + '\n')

# Approach 2: Category stats by Domestic/Foreign (StatType=7)
print('=== Approach 2: CategoryOwnershipPct (StatType=7) ===')
try:
    df2 = rd.get_data(
        universe=test_rics,
        fields=[
            'TR.CategoryOwnershipPct.date',
            'TR.CategoryOwnershipPct',
            'TR.InstrStatCatSharesHeld',
            'TR.CategoryInvestorCount',
            'TR.InstrStatTypeValue',
        ],
        parameters={'StatType': 7, 'SDate': '2020-01-01', 'EDate': '2020-12-31', 'Frq': 'M'}
    )
    print(f'Shape: {df2.shape}')
    print(f'Columns: {list(df2.columns)}')
    print(f'\nUnique categories (InstrStatTypeValue):')
    print(df2.iloc[:, -1].value_counts().to_string())
    print(f'\nNon-null counts:')
    print(df2.notna().sum().to_string())
    print(f'\nSample rows:')
    display(df2.head(30))
except Exception as e:
    print(f'FAILED: {e}')

print('\n' + '='*65 + '\n')

# Approach 3: Try StatType=2 (sometimes domestic/foreign split)
print('=== Approach 3: CategoryOwnershipPct (StatType=2) ===')
try:
    df3 = rd.get_data(
        universe=test_rics,
        fields=[
            'TR.CategoryOwnershipPct.date',
            'TR.CategoryOwnershipPct',
            'TR.InstrStatCatSharesHeld',
            'TR.CategoryInvestorCount',
            'TR.InstrStatTypeValue',
        ],
        parameters={'StatType': 2, 'SDate': '2020-01-01', 'EDate': '2020-12-31', 'Frq': 'M'}
    )
    print(f'Shape: {df3.shape}')
    print(f'Columns: {list(df3.columns)}')
    print(f'\nUnique categories (InstrStatTypeValue):')
    print(df3.iloc[:, -1].value_counts().to_string())
    print(f'\nNon-null counts:')
    print(df3.notna().sum().to_string())
    print(f'\nSample rows:')
    display(df3.head(30))
except Exception as e:
    print(f'FAILED: {e}')

=== Approach 1: CategoryOwnershipPct by Location (StatType=3) ===
Shape: (292, 6)
Columns: ['Instrument', 'Date', 'Category Percent Of Traded Shares', 'Category Investor Shares Held', 'Category Number of Investors', 'Investor Statistics Category Value']

Unique categories (InstrStatTypeValue):
Investor Statistics Category Value
Asia / Pacific    60
Europe            60
North America     60
Africa            60
Middle East       52

Non-null counts:
Instrument                            292
Date                                  292
Category Percent Of Traded Shares     292
Category Investor Shares Held         292
Category Number of Investors          292
Investor Statistics Category Value    292

Sample rows:


,Instrument,Date,Category Percent Of Traded Shares,Category Investor Shares Held,Category Number of Investors,Investor Statistics Category Value
0,BBCA.JK,2020-12-01,62.1859,75892322325,131,Asia / Pacific
1,BBCA.JK,2020-12-01,8.6838,10598341435,155,Europe
2,BBCA.JK,2020-12-01,8.3517,10192803240,124,North America
3,BBCA.JK,2020-12-01,0.5622,686117650,2,Middle East
4,BBCA.JK,2020-12-01,0.0232,28300785,2,Africa
5,BBCA.JK,2020-11-01,63.0644,76965048035,129,Asia / Pacific
6,BBCA.JK,2020-11-01,8.4497,10312113165,122,North America
7,BBCA.JK,2020-11-01,7.2473,8844803115,146,Europe
8,BBCA.JK,2020-11-01,0.6434,785190000,2,Middle East
9,BBCA.JK,2020-11-01,0.0254,31056160,2,Africa




=== Approach 2: CategoryOwnershipPct (StatType=7) ===
Shape: (480, 6)
Columns: ['Instrument', 'Date', 'Category Percent Of Traded Shares', 'Category Investor Shares Held', 'Category Number of Investors', 'Investor Statistics Category Value']

Unique categories (InstrStatTypeValue):
Investor Statistics Category Value
Holdings by Strategic Entities                     60
Holdings by Strategic Entities less Individuals    60
Holdings by Domestic Investors                     60
Holdings by Holding Companies and Corporations     60
Holdings by Foreign Investors                      60
Holdings by Institutions                           60
Holdings by Government                             60
Holdings by Insiders                               60

Non-null counts:
Instrument                            480
Date                                  480
Category Percent Of Traded Shares     480
Category Investor Shares Held         480
Category Number of Investors          480
Investor Statistics 

,Instrument,Date,Category Percent Of Traded Shares,Category Investor Shares Held,Category Number of Investors,Investor Statistics Category Value
0,BBCA.JK,2020-12-01,58.5836,71496742445,20,Holdings by Strategic Entities
1,BBCA.JK,2020-12-01,57.6996,70418130500,6,Holdings by Strategic Entities less Individuals
2,BBCA.JK,2020-12-01,57.681,70395099445,18,Holdings by Domestic Investors
3,BBCA.JK,2020-12-01,56.8089,69330989500,4,Holdings by Holding Companies and Corporations
4,BBCA.JK,2020-12-01,22.1258,27002785990,396,Holdings by Foreign Investors
5,BBCA.JK,2020-12-01,21.1771,25844888570,393,Holdings by Institutions
6,BBCA.JK,2020-12-01,2.7391,3342934785,5,Holdings by Government
7,BBCA.JK,2020-12-01,0.884,1078611945,14,Holdings by Insiders
8,BBCA.JK,2020-11-01,59.1835,72228722190,19,Holdings by Strategic Entities
9,BBCA.JK,2020-11-01,58.7533,71703722190,18,Holdings by Domestic Investors




=== Approach 3: CategoryOwnershipPct (StatType=2) ===
Shape: (883, 6)
Columns: ['Instrument', 'Date', 'Category Percent Of Traded Shares', 'Category Investor Shares Held', 'Category Number of Investors', 'Investor Statistics Category Value']

Unique categories (InstrStatTypeValue):
Investor Statistics Category Value
GARP                      60
Deep Value                60
Yield                     60
Core Growth               60
Hedge Fund                60
Mixed Style               60
Income Value              60
Aggres. Gr.               60
Growth                    60
Index                     60
Core Value                60
Global Macro              58
Specialty                 49
Sector Specific           45
Emerg. Mkts.              36
Long/Short                19
VC/Private Equity          9
Emerging Markets Hedge     7

Non-null counts:
Instrument                            883
Date                                  883
Category Percent Of Traded Shares     883
Category Inves

,Instrument,Date,Category Percent Of Traded Shares,Category Investor Shares Held,Category Number of Investors,Investor Statistics Category Value
0,BBCA.JK,2020-12-01,7.32,8933035725,75,GARP
1,BBCA.JK,2020-12-01,4.4124,5385096690,73,Core Growth
2,BBCA.JK,2020-12-01,3.2534,3970718230,67,Core Value
3,BBCA.JK,2020-12-01,2.8038,3421893455,28,Index
4,BBCA.JK,2020-12-01,1.1771,1436413855,35,Growth
5,BBCA.JK,2020-12-01,0.3229,394190185,8,Aggres. Gr.
6,BBCA.JK,2020-12-01,0.2242,273518240,13,Deep Value
7,BBCA.JK,2020-12-01,0.1901,231941000,2,Mixed Style
8,BBCA.JK,2020-12-01,0.0451,54969085,9,Hedge Fund
9,BBCA.JK,2020-12-01,0.0034,4239180,3,Income Value


## Download all chunks

If the diagnostic above shows different columns than expected, update `clean_inst_batch()` before running this. Each chunk saves to `inst_own_chunk_N.csv`. Existing chunks are skipped.

In [5]:
for chunk_idx, (chunk_start, chunk_end) in enumerate(CHUNKS):
    chunk_file = f'{CHUNK_PREFIX}_{chunk_idx}.csv'
    
    if os.path.exists(chunk_file):
        existing = pd.read_csv(chunk_file)
        print(f'\nChunk {chunk_idx} ({chunk_start.date()} to {chunk_end.date()}): '
              f'ALREADY DONE — {len(existing):,} rows, '
              f'{existing["Instrument"].nunique()} stocks')
        continue
    
    print(f'\n{"="*65}')
    print(f'  Chunk {chunk_idx}: {chunk_start.date()} to {chunk_end.date()}')
    print(f'{"="*65}')
    
    frames = []
    permanently_failed = []
    total_batches = (len(all_rics) + BATCH_SIZE - 1) // BATCH_SIZE
    
    for i in range(0, len(all_rics), BATCH_SIZE):
        batch_rics = all_rics[i:i + BATCH_SIZE]
        batch_num = i // BATCH_SIZE + 1
        
        df_raw, err = fetch_inst_batch(batch_rics, chunk_start, chunk_end)
        
        if df_raw is not None:
            df_clean = clean_inst_batch(df_raw)
            if df_clean is not None and not df_clean.empty:
                frames.append(df_clean)
        elif err:
            permanently_failed.append({
                'rics': batch_rics, 'error': err
            })
        
        if batch_num % 10 == 0 or batch_num == total_batches:
            print(f'  Batch {batch_num}/{total_batches} — '
                  f'{len(frames)} successful, {len(permanently_failed)} failed')
        
        time.sleep(SLEEP)
    
    if frames:
        chunk_df = pd.concat(frames, ignore_index=True)
        chunk_df = chunk_df.drop_duplicates(subset=['Instrument', 'Date', 'Category'], keep='first')
        chunk_df = chunk_df.sort_values(['Instrument', 'Date', 'Category']).reset_index(drop=True)
        chunk_df.to_csv(chunk_file, index=False)
        print(f'\n  Saved {chunk_file}: {len(chunk_df):,} rows, '
              f'{chunk_df["Instrument"].nunique()} stocks')
    else:
        print(f'\n  No data for chunk {chunk_idx}!')
    
    if permanently_failed:
        fail_file = f'{CHUNK_PREFIX}_{chunk_idx}_failed.json'
        with open(fail_file, 'w') as f:
            json.dump(permanently_failed, f, indent=2)
        print(f'  {len(permanently_failed)} failed batches saved to {fail_file}')

print(f'\n{"="*65}')
print('  ALL CHUNKS COMPLETE')
print(f'{"="*65}')


  Chunk 0: 2012-01-01 to 2014-12-31
  Batch 10/100 — 10 successful, 0 failed
  Batch 20/100 — 20 successful, 0 failed
  Batch 30/100 — 30 successful, 0 failed
  Batch 40/100 — 40 successful, 0 failed
  Batch 50/100 — 50 successful, 0 failed
  Batch 60/100 — 60 successful, 0 failed
  Batch 70/100 — 70 successful, 0 failed
  Batch 80/100 — 79 successful, 0 failed
  Batch 90/100 — 88 successful, 0 failed
  Batch 100/100 — 97 successful, 0 failed

  Saved inst_own_chunk_0.csv: 58,487 rows, 382 stocks

  Chunk 1: 2015-01-01 to 2017-12-31
  Batch 10/100 — 10 successful, 0 failed
  Batch 20/100 — 20 successful, 0 failed
  Batch 30/100 — 30 successful, 0 failed
  Batch 40/100 — 40 successful, 0 failed
  Batch 50/100 — 50 successful, 0 failed
  Batch 60/100 — 60 successful, 0 failed
  Batch 70/100 — 70 successful, 0 failed
  Batch 80/100 — 80 successful, 0 failed
  Batch 90/100 — 90 successful, 0 failed
  Batch 100/100 — 99 successful, 0 failed

  Saved inst_own_chunk_1.csv: 96,387 rows, 578 s

## Retry failures

In [6]:
import glob as globmod

fail_files = sorted(globmod.glob(f'{CHUNK_PREFIX}_*_failed.json'))
if not fail_files:
    print('No failed batches to retry!')
else:
    for ff in fail_files:
        parts = os.path.basename(ff).replace('.json', '').split('_')
        chunk_idx = int(parts[3])  # inst_own_chunk_N_failed.json
        chunk_start, chunk_end = CHUNKS[chunk_idx]
        
        with open(ff) as f:
            failures = json.load(f)
        
        print(f'\nRetrying {len(failures)} failed batches from chunk {chunk_idx}...')
        recovered = []
        still_failed = []
        
        for fail in failures:
            df_raw, err = fetch_inst_batch(fail['rics'], chunk_start, chunk_end, retries=5)
            if df_raw is not None:
                df_clean = clean_inst_batch(df_raw)
                if df_clean is not None and not df_clean.empty:
                    recovered.append(df_clean)
                    print(f'  Recovered: {fail["rics"][:3]}...')
            elif err:
                still_failed.append(fail)
            time.sleep(SLEEP * 2)
        
        if recovered:
            rec_df = pd.concat(recovered, ignore_index=True)
            rec_df = rec_df.drop_duplicates(subset=['Instrument', 'Date', 'Category'], keep='first')
            
            chunk_file = f'{CHUNK_PREFIX}_{chunk_idx}.csv'
            if os.path.exists(chunk_file):
                existing = pd.read_csv(chunk_file, parse_dates=['Date'])
                combined = pd.concat([existing, rec_df], ignore_index=True)
                combined = combined.drop_duplicates(subset=['Instrument', 'Date', 'Category'], keep='first')
                combined.to_csv(chunk_file, index=False)
            else:
                rec_df.to_csv(chunk_file, index=False)
            print(f'  +{len(rec_df):,} rows recovered')
        
        if still_failed:
            with open(ff, 'w') as f:
                json.dump(still_failed, f, indent=2)
            print(f'  {len(still_failed)} still failed')
        else:
            os.remove(ff)
            print(f'  All recovered!')
    
    print('\nIf batches were recovered, re-run the Combine cell below.')

No failed batches to retry!


## Combine all chunks

In [7]:
all_chunks = []
for chunk_idx in range(len(CHUNKS)):
    chunk_file = f'{CHUNK_PREFIX}_{chunk_idx}.csv'
    if os.path.exists(chunk_file):
        df = pd.read_csv(chunk_file, parse_dates=['Date'])
        all_chunks.append(df)
        print(f'  Loaded {chunk_file}: {len(df):,} rows, {df["Instrument"].nunique()} stocks')
    else:
        print(f'  MISSING: {chunk_file}')

raw = pd.concat(all_chunks, ignore_index=True)
raw = raw.drop_duplicates(subset=['Instrument', 'Date', 'Category'], keep='first')
raw = raw.sort_values(['Instrument', 'Date', 'Category']).reset_index(drop=True)

print(f'\nCombined raw: {len(raw):,} rows, {raw["Instrument"].nunique()} stocks')
print(f'Date range: {raw["Date"].min().date()} to {raw["Date"].max().date()}')
print(f'\nCategories:\n{raw["Category"].value_counts().to_string()}')

# Pivot: one row per Instrument-Date with Foreign/Domestic/Institutions as columns
KEEP_CATS = ['Holdings by Foreign Investors', 'Holdings by Domestic Investors', 'Holdings by Institutions']
filtered = raw[raw['Category'].isin(KEEP_CATS)].copy()

inst = filtered.pivot_table(
    index=['Instrument', 'Date'],
    columns='Category',
    values='Pct_Held',
    aggfunc='first'
).reset_index()

inst.columns.name = None
inst = inst.rename(columns={
    'Holdings by Foreign Investors': 'Foreign_Pct',
    'Holdings by Domestic Investors': 'Domestic_Pct',
    'Holdings by Institutions': 'Institutional_Pct',
})

inst = inst.sort_values(['Instrument', 'Date']).reset_index(drop=True)

print(f'\nPivoted: {len(inst):,} rows, {inst["Instrument"].nunique()} stocks')
print(f'\nField coverage (non-null %):')
for col in ['Foreign_Pct', 'Domestic_Pct', 'Institutional_Pct']:
    if col in inst.columns:
        pct = inst[col].notna().mean() * 100
        print(f'  {col}: {pct:.1f}%')

print(f'\nForeign_Pct distribution:')
print(inst['Foreign_Pct'].describe().to_string())

  Loaded inst_own_chunk_0.csv: 58,487 rows, 382 stocks
  Loaded inst_own_chunk_1.csv: 96,387 rows, 578 stocks
  Loaded inst_own_chunk_2.csv: 136,279 rows, 737 stocks
  Loaded inst_own_chunk_3.csv: 163,528 rows, 875 stocks
  Loaded inst_own_chunk_4.csv: 122,473 rows, 920 stocks

Combined raw: 577,154 rows, 967 stocks
Date range: 2012-01-01 to 2025-12-01

Categories:
Category
Holdings by Strategic Entities                     100859
Holdings by Strategic Entities less Individuals     96715
Holdings by Holding Companies and Corporations      95203
Holdings by Domestic Investors                      90345
Holdings by Foreign Investors                       63541
Holdings by Insiders                                61468
Holdings by Institutions                            52751
Holdings by Government                              16272

Pivoted: 96,646 rows, 965 stocks

Field coverage (non-null %):
  Foreign_Pct: 65.7%
  Domestic_Pct: 93.5%
  Institutional_Pct: 54.6%

Foreign_Pct distribution

## Save final file

In [8]:
inst.to_csv(f'{DATA_DIR}/idx_institutional_ownership.csv', index=False)
print(f'Saved {DATA_DIR}/idx_institutional_ownership.csv: {len(inst):,} rows, {inst["Instrument"].nunique()} stocks')

Saved ../data/idx_institutional_ownership.csv: 96,646 rows, 965 stocks


## Quality check

In [9]:
print('=== QUALITY CHECK ===\n')

# Blue-chip spot check
blue_chips = ['BBCA.JK', 'BBRI.JK', 'BMRI.JK', 'TLKM.JK', 'ASII.JK',
              'UNVR.JK', 'HMSP.JK', 'GGRM.JK', 'ICBP.JK', 'KLBF.JK']

inst_rics = set(inst['Instrument'].unique())
print('Blue-chip coverage (Foreign % = share held by foreign investors):')
for bc in blue_chips:
    if bc in inst_rics:
        sub = inst[inst.Instrument == bc]
        avg_f = sub['Foreign_Pct'].mean()
        avg_d = sub['Domestic_Pct'].mean()
        print(f'  {bc}: {len(sub)} obs, avg foreign = {avg_f:.1f}%, avg domestic = {avg_d:.1f}%')
    else:
        print(f'  {bc}: MISSING')

print()

# Coverage vs master panel
mp_rics = set(pd.read_csv(f'{DATA_DIR}/idx_master_panel.csv', usecols=['Instrument'])['Instrument'].unique())
overlap = inst_rics & mp_rics
print(f'Panel stocks with ownership data: {len(overlap)} / {len(mp_rics)} ({len(overlap)/len(mp_rics)*100:.1f}%)')

print()

# Stocks per year
inst_copy = inst.copy()
inst_copy['Year'] = inst_copy['Date'].dt.year
print('Stocks with ownership data by year:')
for y in range(2012, 2026):
    n = inst_copy[inst_copy.Year == y]['Instrument'].nunique()
    if n > 0:
        print(f'  {y}: {n}')

print()

# Distribution
for col in ['Foreign_Pct', 'Domestic_Pct', 'Institutional_Pct']:
    if col in inst.columns:
        print(f'{col} distribution:')
        print(inst[col].describe().to_string())
        print()

=== QUALITY CHECK ===

Blue-chip coverage (Foreign % = share held by foreign investors):
  BBCA.JK: 168 obs, avg foreign = 17.3%, avg domestic = 54.5%
  BBRI.JK: 168 obs, avg foreign = 18.6%, avg domestic = 57.8%
  BMRI.JK: 168 obs, avg foreign = 17.4%, avg domestic = 61.3%
  TLKM.JK: 168 obs, avg foreign = 17.4%, avg domestic = 53.5%
  ASII.JK: 168 obs, avg foreign = 70.8%, avg domestic = 0.0%
  UNVR.JK: 168 obs, avg foreign = 89.3%, avg domestic = 0.0%
  HMSP.JK: 168 obs, avg foreign = 0.7%, avg domestic = 93.6%
  GGRM.JK: 168 obs, avg foreign = 4.5%, avg domestic = 76.2%
  ICBP.JK: 168 obs, avg foreign = 5.7%, avg domestic = 80.5%
  KLBF.JK: 168 obs, avg foreign = 12.0%, avg domestic = 56.7%

Panel stocks with ownership data: 961 / 967 (99.4%)

Stocks with ownership data by year:
  2012: 277
  2013: 300
  2014: 329
  2015: 377
  2016: 420
  2017: 575
  2018: 630
  2019: 679
  2020: 725
  2021: 763
  2022: 804
  2023: 859
  2024: 894
  2025: 907

Foreign_Pct distribution:
count    63